## Sample Read Delta table on Minio

In [1]:
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

from dotenv import load_dotenv
import os

In [ ]:
load_dotenv()

MINIO_CONTAINER=os.getenv("MINIO_CONTAINER")
MINIO_USER=os.getenv("MINIO_USER")
MINIO_PASSWORD=os.getenv("MINIO_PASSWORD")

In [4]:
conf = SparkConf()

conf.setAppName("Example read delta table") # nome da aplicação Spark, útil para logs
conf.set("spark.hadoop.fs.s3a.endpoint",f"http://{MINIO_CONTAINER}:9000") # Container and Port from MinIO
conf.set("spark.hadoop.fs.s3a.access.key", f"{MINIO_USER}") # Login from MinIO
conf.set("spark.hadoop.fs.s3a.secret.key", f"{MINIO_PASSWORD}") # Password from MinIO
conf.set("spark.hadoop.fs.s3a.path.style.access", True)
conf.set("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") # Talk to Hadoop/Spark to use new conector S3A
conf.set("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") # How to credentials are acess via config(access key + secret)
conf.set("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") # active extension from Delta Lake
conf.set("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") # Change the standard catalog from spark to Delta 
conf.set("hive.metastore.uris", "thrift://metastore:9083") # Connect to Hive Metastore external

spark = SparkSession.builder.config(conf=conf).enableHiveSupport().getOrCreate()


## reading file from MinIO S3 bronze bucket in Delta format

In [5]:
dataFrame = spark.read.format("delta").load("s3a://bronze/delta_example")

In [6]:
dataFrame.show()

+----------+---------+------+------+
|first_name|last_name|Gender|salary|
+----------+---------+------+------+
|   Rodrigo| Maturino|     M|  4000|
|   Eduarda|   Santos|     F|  5000|
|    Rafael|    Souza|     M|  4500|
|     Luigi|    Bispo|     M|  6000|
+----------+---------+------+------+

